# Transformer Encoder + Positional Encoding — IRB2400 Kinematics

## 이 노트북이 하는 일
- `tr_encoder_kinematics.ipynb` (baseline, PE 없음) 에 **Positional Encoding 을 추가** 한 버전
- 목표는 동일: 관절각 시퀀스 → 다음 포즈 (x, y, z, yaw, pitch, roll)

## 두 개의 실험
1. **[A] PE 추가만** — baseline 과 거의 동일하지만 sinusoidal PE 를 입력에 더함
2. **[B] 하이퍼파라미터 튜닝** — 시퀀스 길이↑, head 수↑, dropout↑, Dense head 크기↑, EarlyStopping 추가

두 실험을 한 노트북에 배치해서 **PE 효과 vs 튜닝 효과** 를 비교할 수 있게 했어.


---
# [A] PE 만 추가한 실험

baseline 코드를 그대로 가져오되 `PositionalEncoding` Layer 하나만 꽂은 구성.


## [A-1] 라이브러리 임포트


In [ ]:
# [A-1] 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LayerNormalization, Dropout,
    GlobalAveragePooling1D, MultiHeadAttention, Add,
)


## [A-2] Google Drive 마운트 (Colab 전용)


In [ ]:
# [A-2] Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')


## [A-3]~[A-5] 데이터 로딩 / 입·출력 / 정규화

baseline 과 완전 동일. 정규화 이유:
- 위치(mm) 와 각도(rad) 의 스케일 차이를 맞춰서 MSE 가 균형있게 동작하도록
- Attention 입력 분포를 안정화


In [ ]:
# [A-3] 데이터 로딩
csv_path = '/content/drive/MyDrive/Colab Notebooks/data/ngv/datasetIRB2400.csv'
df = pd.read_csv(csv_path)
print("데이터셋 로딩 완료")
print(df.head())

# [A-4] 입력 및 출력 설정
X = df[['q1_in', 'q2_in', 'q3_in', 'q4_in', 'q5_in', 'q6_in']].values
y = df[['x', 'y', 'z', 'yaw', 'pitch', 'roll']].values

# [A-5] 정규화 (입·출력 각각 별도 scaler)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)


## [A-6] 시퀀스 생성

baseline 과 동일하게 sliding window.
`time_steps=10` 으로 고정해서 PE 효과를 순수하게 비교.


In [ ]:
# [A-6] 시퀀스 생성
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i + time_steps])   # (T, 6)
        ys.append(y[i + time_steps])      # (6,)
    return np.array(Xs), np.array(ys)

time_steps = 10
X_seq, y_seq = create_sequences(X_scaled, y_scaled, time_steps)

# [A-7] 학습/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


## [A-8] Positional Encoding — 왜 넣는가

Transformer 의 self-attention 은 **permutation-invariant** (입력 순서를 바꿔도 결과 같음).
즉 시퀀스의 "몇 번째 스텝인지" 구분 못 함. PE 가 그걸 해결.

### 왜 sin/cos 방식인가 (고전 방식)
- **학습 파라미터 0개** → 과적합 위험 없고, 시퀀스 길이가 바뀌어도 작동
- 서로 다른 주파수의 sin/cos 으로 위치를 **주기 조합** 으로 표현 → attention 이 임의 두 위치 차이를 선형 변환으로 복원 가능 (원 논문 "Attention Is All You Need" 의도)

### 수식
```
PE(pos, 2k)   = sin(pos / 10000^(2k/d))
PE(pos, 2k+1) = cos(pos / 10000^(2k/d))
```
짝수 차원엔 sin, 홀수 차원엔 cos. `angle_rates` 는 차원을 따라 지수적으로 감쇠하는 주파수.


In [ ]:
# [A-8] Positional Encoding Layer (수정됨 — 원본 변수명 유지)
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, inputs):
        # inputs: (batch, T, d_model)
        seq_len = tf.shape(inputs)[1]             # T — 동적 shape 로 읽어야 그래프 모드에서도 안전
        d_model = inputs.shape[-1]                # 정적 shape (모델 설계 시 고정)

        # 위치 및 차원 인덱스
        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)   # (T, 1)
        i   = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)   # (1, d)

        # 원 논문의 주파수 스케일링
        angle_rates = 1 / tf.pow(10000.0, (2 * (i // 2)) / d_model)
        angle_rads  = pos * angle_rates                               # (T, d)

        # 짝수엔 sin, 홀수엔 cos
        sines   = tf.sin(angle_rads[:, 0::2])
        cosines = tf.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)           # (T, d)

        pos_encoding = pos_encoding[tf.newaxis, ...]                  # (1, T, d) — 배치 broadcast
        return inputs + pos_encoding                                  # 입력에 위치 정보 주입


## [A-9] Transformer Encoder (Pre-Norm)

baseline 과 동일한 블록. LN → MHA → Add → LN → FFN → Add.


In [ ]:
# [A-9] Transformer Encoder 정의
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = Add()([x, inputs])

    x = LayerNormalization(epsilon=1e-6)(x)
    x_ff = Dense(ff_dim, activation='relu')(x)
    x_ff = Dense(inputs.shape[-1])(x_ff)
    return Add()([x, x_ff])


## [A-10] 모델 구성 — baseline + PE

baseline 과의 **유일한 차이는 맨 앞 `PositionalEncoding()`** 한 줄.
나머지는 전부 동일해서 성능 차이를 PE 덕분으로 귀속할 수 있음.


In [ ]:
# [A-10] 모델 구성
input_shape = (time_steps, X_train.shape[2])
inputs = Input(shape=input_shape)

# ★ baseline 과의 차이는 여기 한 줄: PE 주입
x = PositionalEncoding()(inputs)

# 이후는 baseline 과 동일
x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(6)(x)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


## [A-11]~[A-16] 학습 · 평가 · 시각화

baseline 과 **똑같은 epochs/batch** 로 학습해서 PE 효과만 비교.


In [ ]:
# [A-11] 모델 학습
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
)

# [A-12] 학습 손실 시각화
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
plt.title("Training and Validation Loss")
plt.show()

# [A-13] 예측 + [A-14] 역정규화
y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test)

# [A-15] MAE (전체)
mae_real = mean_absolute_error(y_true, y_pred)
print(f"\n실제 단위 기준 MAE: {mae_real:.4f}")

# [A-16] 차원별 MAE
columns = ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
mae_each = np.mean(np.abs(y_true - y_pred), axis=0)
for name, value in zip(columns, mae_each):
    print(f"{name} MAE: {value:.4f}")


---
# [B] 하이퍼파라미터 튜닝 강화 버전

[A] 대비 바뀐 점 한눈에:

| 항목            | [A] (baseline + PE) | [B] (튜닝 강화) | 왜                                      |
|-----------------|---------------------|-----------------|-----------------------------------------|
| `time_steps`    | 10                  | **30**          | 더 긴 시간 의존성 포착                   |
| `num_heads`     | 4                   | **8**           | 더 다양한 'attention 관점'               |
| `head_size`     | 64                  | 32              | head 수가 늘어난 만큼 차원 축소 → 총 dim 256 유지 |
| `ff_dim`        | 128                 | **256**         | FFN 표현력 강화                          |
| `dropout`       | 0.1                 | **0.2** + FFN 내부 추가 | 시퀀스 길어지면 과적합↑ → 정규화 강화 |
| Dense head      | 64                  | **128 + Dropout(0.3)** | head 표현력 + 정규화                |
| `batch_size`    | 32                  | **64**          | 긴 시퀀스로 메모리 여유 ↓ → 배치 키워 gradient 안정 |
| `epochs`        | 10                  | **20 + EarlyStopping(patience=10)** | 충분한 상한 + 과적합 시 조기 종료 |
| 평가지표        | MAE                 | **MAE + RMSE + R²** | 다각도 해석                         |
| 시각화          | Loss                | Loss + True vs Pred(x) | 예측 품질 시각적 진단              |


## [B-1] 임포트 & 데이터 준비

`EarlyStopping` 을 새로 추가 임포트. 나머지는 동일.


In [ ]:
# [B-1] 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LayerNormalization, Dropout,
    GlobalAveragePooling1D, MultiHeadAttention, Add,
)
from tensorflow.keras.callbacks import EarlyStopping     # [B] 에서 추가


In [ ]:
# [B-2] 데이터 로딩 & 정규화 (위와 동일 로직)
csv_path = '/content/drive/MyDrive/Colab Notebooks/data/ngv/datasetIRB2400.csv'
df = pd.read_csv(csv_path)

X = df[['q1_in', 'q2_in', 'q3_in', 'q4_in', 'q5_in', 'q6_in']].values
y = df[['x', 'y', 'z', 'yaw', 'pitch', 'roll']].values

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)


## [B-3] 시퀀스 길이 확장 (`time_steps=30`)

왜 10 → 30?
- 로봇팔 궤적은 짧은 구간(10 스텝) 이내에 대부분 비슷한 방향으로 움직임 → self-attention 이 거의 identity
- 30 스텝이면 방향 전환/가감속 패턴까지 보게 돼서 **시간적 문맥** 이 풍부해짐
- 다만 시퀀스 길어질수록 attention O(T²) 비용 + 과적합 위험 ↑ → 그래서 dropout/정규화도 같이 강화


In [ ]:
# [B-3] 시퀀스 생성 (더 긴 윈도우)
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i + time_steps])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 30                                    # [A] 의 10 → 30 으로 확장
X_seq, y_seq = create_sequences(X_scaled, y_scaled, time_steps)
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)


## [B-4] PositionalEncoding (위 [A-8] 과 동일)


In [ ]:
# [B-4] PositionalEncoding (개선된 PositionalEncoding 이라 주석 달려있으나 로직은 [A] 와 동일)
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        d_model = inputs.shape[-1]
        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        i   = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)
        angle_rates = 1 / tf.pow(10000.0, (2 * (i // 2)) / d_model)
        angle_rads  = pos * angle_rates
        sines   = tf.sin(angle_rads[:, 0::2])
        cosines = tf.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)[tf.newaxis, ...]
        return inputs + pos_encoding


## [B-5] Encoder 블록 — FFN 에도 Dropout 추가

[A] 의 encoder 와 거의 같지만 **FFN hidden 뒤에 Dropout 추가**.
시퀀스가 길어지고 ff_dim 도 커졌기 때문에 FFN 자체의 과적합을 막는 용도.


In [ ]:
# [B-5] 개선된 Transformer encoder — FFN 안에도 dropout 추가
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.2):
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = Add()([x, inputs])

    x = LayerNormalization(epsilon=1e-6)(x)
    x_ff = Dense(ff_dim, activation='relu')(x)
    x_ff = Dropout(dropout)(x_ff)                       # ★ [A] 에는 없던 추가 dropout
    x_ff = Dense(inputs.shape[-1])(x_ff)
    return Add()([x, x_ff])


## [B-6] 모델 구성 — 머리 많고 얇은 head, 큰 FFN, Dense head 강화

### head_size=32, num_heads=8 으로 바꾼 이유
- 총 attention 차원 = head_size × num_heads = 32 × 8 = **256** (= [A] 의 64×4 와 같음)
- **차원은 유지하면서 head 수만 늘림** → 더 다양한 관점으로 시퀀스 보기
- 각 head 가 얇아져도 수가 많아지면 앙상블 효과


In [ ]:
# [B-6] 모델 구성
input_shape = (time_steps, X_train.shape[2])
inputs = Input(shape=input_shape)
x = PositionalEncoding()(inputs)

# head_size=32, num_heads=8 → 총 256 dim, 더 다양한 관점
x = transformer_encoder(x, head_size=32, num_heads=8, ff_dim=256, dropout=0.2)
x = transformer_encoder(x, head_size=32, num_heads=8, ff_dim=256, dropout=0.2)

x = GlobalAveragePooling1D()(x)
x = Dense(128, activation='relu')(x)                   # [A] 의 64 → 128
x = Dropout(0.3)(x)                                    # head 정규화
outputs = Dense(6)(x)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


## [B-7] 학습 — EarlyStopping + `restore_best_weights`

### patience=10 인 이유
Transformer 는 처음 몇 epoch 는 불안정하게 움직임. patience 가 너무 짧으면 진짜 수렴 전에 멈춰버림.
Epoch 상한은 20 이라 patience 10 이면 충분히 여유롭게 본 뒤 종료.

### restore_best_weights=True 중요!
종료 시점 가중치는 과적합된 상태일 수 있음. val_loss 기준 **가장 좋았던 가중치** 로 되돌려야 평가/배포 시 실제 성능이 좋음.


In [ ]:
# [B-7] 학습
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=64,                                     # [A] 의 32 → 64
    callbacks=[early_stop],
    verbose=1,
)


## [B-8] 평가 — 세 가지 지표

| 지표 | 의미 | 왜                      |
|------|------|------------------------|
| MAE  | 평균 절대 오차 | 실제 단위로 직관적 해석 |
| RMSE | 평균 제곱근 오차 | 큰 오차에 민감 → outlier 진단 |
| R²   | 분산 설명도     | 스케일 무관 성능지표, 1에 가까울수록 좋음 |


In [ ]:
# [B-8] 예측 + 역정규화 + 평가
y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test)

mae_real  = mean_absolute_error(y_true, y_pred)
rmse_real = np.sqrt(mean_squared_error(y_true, y_pred))
r2        = r2_score(y_true, y_pred)

print(f"\nMAE : {mae_real:.4f}")
print(f"RMSE: {rmse_real:.4f}")
print(f"R^2 : {r2:.4f}")

columns = ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
mae_each = np.mean(np.abs(y_true - y_pred), axis=0)
for name, value in zip(columns, mae_each):
    print(f"{name} MAE: {value:.4f}")


## [B-9] 시각화

1. **Loss 곡선** — 과적합 여부 확인
2. **x 좌표 예측 vs 실제** — MAE 숫자만으로는 안 보이는 패턴 (lag, bias, spike) 확인


In [ ]:
# [B-9] 시각화: Loss
plt.figure()
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.title("Loss Curve"); plt.grid(True)
plt.show()

# x 좌표 예측 vs 실제
plt.figure()
plt.plot(y_true[:, 0], label='True x')
plt.plot(y_pred[:, 0], label='Pred x')
plt.legend(); plt.title("True vs Predicted - x")
plt.xlabel("Sample Index"); plt.ylabel("Position")
plt.grid(True); plt.show()
